In [57]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE THE SILVER AUDIT
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

SILVER_SCHEMA = "silver"

EXPECTED_ROW_COUNTS = {
    "patients": 1163,
    "encounters": 61459,
    "claims": 117889,
    "claims_transactions": 711238,
    "payers": 10,
    "payer_transitions": 53101
}

SILVER_TABLES = list(
    EXPECTED_ROW_COUNTS.keys()
)

print("Silver data-quality audit configured.")
print(f"Schema: {SILVER_SCHEMA}")
print(f"Tables to audit: {len(SILVER_TABLES)}")

for table_name in SILVER_TABLES:
    print(
        f"  {SILVER_SCHEMA}.{table_name}: "
        f"expected {EXPECTED_ROW_COUNTS[table_name]:,} rows"
    )


StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 59, Finished, Available, Finished, False)

Silver data-quality audit configured.
Schema: silver
Tables to audit: 6
  silver.patients: expected 1,163 rows
  silver.encounters: expected 61,459 rows
  silver.claims: expected 117,889 rows
  silver.claims_transactions: expected 711,238 rows
  silver.payers: expected 10 rows
  silver.payer_transitions: expected 53,101 rows


In [58]:
# ============================================================
# 2. VERIFY SILVER TABLE AVAILABILITY AND ROW COUNTS
# ============================================================

silver_row_count_results = []

for table_name, expected_count in EXPECTED_ROW_COUNTS.items():

    full_table_name = (
        f"{SILVER_SCHEMA}.{table_name}"
    )

    table_exists = spark.catalog.tableExists(
        full_table_name
    )

    if table_exists:
        actual_count = (
            spark.table(full_table_name).count()
        )

        audit_status = (
            "PASS"
            if actual_count == expected_count
            else "FAIL"
        )

        count_difference = (
            actual_count - expected_count
        )

    else:
        actual_count = None
        count_difference = None
        audit_status = "MISSING"

    silver_row_count_results.append(
        (
            full_table_name,
            table_exists,
            expected_count,
            actual_count,
            count_difference,
            audit_status
        )
    )


row_count_audit_schema = T.StructType([
    T.StructField(
        "table_name",
        T.StringType(),
        False
    ),
    T.StructField(
        "table_exists",
        T.BooleanType(),
        False
    ),
    T.StructField(
        "expected_row_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "actual_row_count",
        T.LongType(),
        True
    ),
    T.StructField(
        "count_difference",
        T.LongType(),
        True
    ),
    T.StructField(
        "audit_status",
        T.StringType(),
        False
    )
])


silver_row_count_audit = (
    spark.createDataFrame(
        silver_row_count_results,
        schema=row_count_audit_schema
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
)

display(
    silver_row_count_audit
    .orderBy("table_name")
)


failed_table_count = (
    silver_row_count_audit
    .filter(
        F.col("audit_status") != "PASS"
    )
    .count()
)

if failed_table_count > 0:
    raise ValueError(
        f"{failed_table_count} Silver table row-count "
        "audits failed. Review the displayed results."
    )

print(
    "All six Silver tables exist and their "
    "row counts match the expected source counts."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 636697ff-bebd-4314-803e-ba276e6c0265)

All six Silver tables exist and their row counts match the expected source counts.


In [59]:
# ============================================================
# 3. VALIDATE SILVER PRIMARY KEYS
# ============================================================

PRIMARY_KEY_CONFIG = {
    "patients": "id",
    "encounters": "encounter_id",
    "claims": "claim_id",
    "claims_transactions": "transaction_id",
    "payers": "payer_id"
}

primary_key_results = []

for table_name, primary_key_column in PRIMARY_KEY_CONFIG.items():

    full_table_name = (
        f"{SILVER_SCHEMA}.{table_name}"
    )

    table_df = spark.table(
        full_table_name
    )

    # Count all rows in the table
    row_count = table_df.count()

    # Count null primary-key values
    null_key_count = (
        table_df
        .filter(
            F.col(primary_key_column).isNull()
        )
        .count()
    )

    # Count distinct non-null primary-key values
    distinct_key_count = (
        table_df
        .select(primary_key_column)
        .distinct()
        .count()
    )

    # Find primary-key values occurring more than once
    duplicate_key_values = (
        table_df
        .groupBy(primary_key_column)
        .count()
        .filter(
            F.col("count") > 1
        )
    )

    duplicate_key_group_count = (
        duplicate_key_values.count()
    )

    duplicate_key_extra_rows = (
        duplicate_key_values
        .agg(
            F.coalesce(
                F.sum(
                    F.col("count") - 1
                ),
                F.lit(0)
            ).alias(
                "duplicate_extra_rows"
            )
        )
        .first()["duplicate_extra_rows"]
    )

    audit_status = (
        "PASS"
        if (
            null_key_count == 0
            and
            duplicate_key_group_count == 0
            and
            distinct_key_count == row_count
        )
        else "FAIL"
    )

    primary_key_results.append(
        (
            full_table_name,
            primary_key_column,
            row_count,
            distinct_key_count,
            null_key_count,
            duplicate_key_group_count,
            duplicate_key_extra_rows,
            audit_status
        )
    )


primary_key_audit_schema = T.StructType([
    T.StructField(
        "table_name",
        T.StringType(),
        False
    ),
    T.StructField(
        "primary_key_column",
        T.StringType(),
        False
    ),
    T.StructField(
        "row_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "distinct_key_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "null_key_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "duplicate_key_groups",
        T.LongType(),
        False
    ),
    T.StructField(
        "duplicate_extra_rows",
        T.LongType(),
        False
    ),
    T.StructField(
        "audit_status",
        T.StringType(),
        False
    )
])


primary_key_audit = (
    spark.createDataFrame(
        primary_key_results,
        schema=primary_key_audit_schema
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
)

display(
    primary_key_audit
    .orderBy("table_name")
)


failed_primary_key_count = (
    primary_key_audit
    .filter(
        F.col("audit_status") != "PASS"
    )
    .count()
)

if failed_primary_key_count > 0:
    raise ValueError(
        f"{failed_primary_key_count} primary-key "
        "audits failed."
    )

print(
    "Primary-key completeness and uniqueness "
    "validations passed for all five keyed tables."
)


StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b6547842-0261-4de5-ab0c-551a13bbede3)

Primary-key completeness and uniqueness validations passed for all five keyed tables.


In [60]:
# ============================================================
# 4. VALIDATE EXACT DUPLICATE BUSINESS ROWS
# ============================================================

AUDIT_METADATA_COLUMNS = {
    "record_hash",
    "source_system",
    "source_table",
    "source_file",
    "silver_processed_timestamp"
}

duplicate_audit_results = []

for table_name in SILVER_TABLES:

    full_table_name = (
        f"{SILVER_SCHEMA}.{table_name}"
    )

    table_df = spark.table(
        full_table_name
    )

    # Exclude processing metadata from the business-row check
    business_columns = [
        column_name
        for column_name in table_df.columns
        if column_name not in AUDIT_METADATA_COLUMNS
    ]

    # Create a deterministic signature for each business row
    table_with_signature = (
        table_df
        .withColumn(
            "_audit_row_signature",
            F.sha2(
                F.to_json(
                    F.struct(
                        *[
                            F.col(column_name)
                            for column_name
                            in business_columns
                        ]
                    ),
                    options={
                        "ignoreNullFields": "false"
                    }
                ),
                256
            )
        )
    )

    # Identify signatures occurring more than once
    duplicate_business_rows = (
        table_with_signature
        .groupBy(
            "_audit_row_signature"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
    )

    duplicate_group_count = (
        duplicate_business_rows.count()
    )

    duplicate_extra_row_count = (
        duplicate_business_rows
        .agg(
            F.coalesce(
                F.sum(
                    F.col("count") - 1
                ),
                F.lit(0)
            ).alias(
                "duplicate_extra_rows"
            )
        )
        .first()["duplicate_extra_rows"]
    )

    audit_status = (
        "PASS"
        if duplicate_group_count == 0
        else "FAIL"
    )

    duplicate_audit_results.append(
        (
            full_table_name,
            len(business_columns),
            duplicate_group_count,
            duplicate_extra_row_count,
            audit_status
        )
    )


duplicate_audit_schema = T.StructType([
    T.StructField(
        "table_name",
        T.StringType(),
        False
    ),
    T.StructField(
        "business_column_count",
        T.IntegerType(),
        False
    ),
    T.StructField(
        "duplicate_groups",
        T.LongType(),
        False
    ),
    T.StructField(
        "duplicate_extra_rows",
        T.LongType(),
        False
    ),
    T.StructField(
        "audit_status",
        T.StringType(),
        False
    )
])


exact_duplicate_audit = (
    spark.createDataFrame(
        duplicate_audit_results,
        schema=duplicate_audit_schema
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
)

display(
    exact_duplicate_audit
    .orderBy("table_name")
)


failed_duplicate_table_count = (
    exact_duplicate_audit
    .filter(
        F.col("audit_status") != "PASS"
    )
    .count()
)

if failed_duplicate_table_count > 0:
    raise ValueError(
        f"{failed_duplicate_table_count} Silver tables "
        "contain exact duplicate business rows."
    )

print(
    "No exact duplicate business rows were detected "
    "in the six Silver tables."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 62, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc31ace7-2431-49ee-8e72-3d2be8bedc01)

No exact duplicate business rows were detected in the six Silver tables.


In [61]:
# ============================================================
# 5. VALIDATE SILVER FOREIGN-KEY RELATIONSHIPS
# ============================================================

FOREIGN_KEY_CONFIG = [
    # Encounter relationships
    {
        "relationship": "encounters_to_patients",
        "source_table": "encounters",
        "source_column": "patient_id",
        "target_table": "patients",
        "target_column": "id",
        "null_allowed": False
    },
    {
        "relationship": "encounters_to_payers",
        "source_table": "encounters",
        "source_column": "payer_id",
        "target_table": "payers",
        "target_column": "payer_id",
        "null_allowed": False
    },

    # Claim relationships
    {
        "relationship": "claims_to_patients",
        "source_table": "claims",
        "source_column": "patient_id",
        "target_table": "patients",
        "target_column": "id",
        "null_allowed": False
    },
    {
        "relationship": "claims_to_encounters",
        "source_table": "claims",
        "source_column": "encounter_id",
        "target_table": "encounters",
        "target_column": "encounter_id",
        "null_allowed": False
    },
    {
        "relationship": "claims_to_primary_payers",
        "source_table": "claims",
        "source_column": "primary_payer_id",
        "target_table": "payers",
        "target_column": "payer_id",
        "null_allowed": True
    },
    {
        "relationship": "claims_to_secondary_payers",
        "source_table": "claims",
        "source_column": "secondary_payer_id",
        "target_table": "payers",
        "target_column": "payer_id",
        "null_allowed": True
    },

    # Claim-transaction relationships
    {
        "relationship": "transactions_to_claims",
        "source_table": "claims_transactions",
        "source_column": "claim_id",
        "target_table": "claims",
        "target_column": "claim_id",
        "null_allowed": False
    },
    {
        "relationship": "transactions_to_patients",
        "source_table": "claims_transactions",
        "source_column": "patient_id",
        "target_table": "patients",
        "target_column": "id",
        "null_allowed": False
    },
    {
        "relationship": "transactions_to_encounters",
        "source_table": "claims_transactions",
        "source_column": "encounter_id",
        "target_table": "encounters",
        "target_column": "encounter_id",
        "null_allowed": False
    },

    # Payer-transition relationships
    {
        "relationship": "payer_transitions_to_patients",
        "source_table": "payer_transitions",
        "source_column": "patient_id",
        "target_table": "patients",
        "target_column": "id",
        "null_allowed": False
    },
    {
        "relationship": "payer_transitions_to_primary_payers",
        "source_table": "payer_transitions",
        "source_column": "payer_id",
        "target_table": "payers",
        "target_column": "payer_id",
        "null_allowed": False
    },
    {
        "relationship": "payer_transitions_to_secondary_payers",
        "source_table": "payer_transitions",
        "source_column": "secondary_payer_id",
        "target_table": "payers",
        "target_column": "payer_id",
        "null_allowed": True
    }
]

foreign_key_results = []

for relationship_config in FOREIGN_KEY_CONFIG:

    relationship_name = (
        relationship_config["relationship"]
    )

    source_table_name = (
        relationship_config["source_table"]
    )
    source_column = (
        relationship_config["source_column"]
    )

    target_table_name = (
        relationship_config["target_table"]
    )
    target_column = (
        relationship_config["target_column"]
    )

    null_allowed = (
        relationship_config["null_allowed"]
    )

    source_df = (
        spark.table(
            f"{SILVER_SCHEMA}.{source_table_name}"
        )
        .select(source_column)
    )

    target_keys = (
        spark.table(
            f"{SILVER_SCHEMA}.{target_table_name}"
        )
        .select(
            F.col(target_column).alias(
                "_valid_target_key"
            )
        )
        .distinct()
    )

    source_row_count = source_df.count()

    null_source_key_count = (
        source_df
        .filter(
            F.col(source_column).isNull()
        )
        .count()
    )

    populated_source_keys = (
        source_df
        .filter(
            F.col(source_column).isNotNull()
        )
    )

    populated_source_key_count = (
        populated_source_keys.count()
    )

    invalid_foreign_keys = (
        populated_source_keys.alias("source")
        .join(
            target_keys.alias("target"),
            F.col(
                f"source.{source_column}"
            )
            ==
            F.col(
                "target._valid_target_key"
            ),
            "left_anti"
        )
    )

    invalid_foreign_key_count = (
        invalid_foreign_keys.count()
    )

    null_rule_passed = (
        null_allowed
        or
        null_source_key_count == 0
    )

    audit_status = (
        "PASS"
        if (
            invalid_foreign_key_count == 0
            and
            null_rule_passed
        )
        else "FAIL"
    )

    foreign_key_results.append(
        (
            relationship_name,
            (
                f"{SILVER_SCHEMA}."
                f"{source_table_name}."
                f"{source_column}"
            ),
            (
                f"{SILVER_SCHEMA}."
                f"{target_table_name}."
                f"{target_column}"
            ),
            null_allowed,
            source_row_count,
            populated_source_key_count,
            null_source_key_count,
            invalid_foreign_key_count,
            audit_status
        )
    )


foreign_key_audit_schema = T.StructType([
    T.StructField("relationship", T.StringType(), False),
    T.StructField("source_key", T.StringType(), False),
    T.StructField("target_key", T.StringType(), False),
    T.StructField("null_allowed", T.BooleanType(), False),
    T.StructField("source_rows", T.LongType(), False),
    T.StructField("populated_source_keys", T.LongType(), False),
    T.StructField("null_source_keys", T.LongType(), False),
    T.StructField("invalid_foreign_keys", T.LongType(), False),
    T.StructField("audit_status", T.StringType(), False)
])


foreign_key_audit = (
    spark.createDataFrame(
        foreign_key_results,
        schema=foreign_key_audit_schema
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
)

display(
    foreign_key_audit
    .orderBy("relationship")
)


failed_foreign_key_count = (
    foreign_key_audit
    .filter(
        F.col("audit_status") != "PASS"
    )
    .count()
)

if failed_foreign_key_count > 0:
    raise ValueError(
        f"{failed_foreign_key_count} foreign-key "
        "relationship audits failed."
    )

print(
    "All intended Silver foreign-key "
    "relationship validations passed."
)

print(
    "claims_transactions.source_patient_insurance_id "
    "was intentionally excluded from payer validation."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a1f615dd-de23-4b28-925a-c1fe58dea8cc)

All intended Silver foreign-key relationship validations passed.
claims_transactions.source_patient_insurance_id was intentionally excluded from payer validation.


In [62]:
# ============================================================
# 6. VALIDATE ENCOUNTER-CLAIM-TRANSACTION COVERAGE
# ============================================================

encounter_keys = (
    spark.table(
        f"{SILVER_SCHEMA}.encounters"
    )
    .select(
        "encounter_id"
    )
    .distinct()
)

claim_keys = (
    spark.table(
        f"{SILVER_SCHEMA}.claims"
    )
    .select(
        "claim_id",
        "encounter_id"
    )
)

transaction_claim_keys = (
    spark.table(
        f"{SILVER_SCHEMA}.claims_transactions"
    )
    .select(
        "claim_id"
    )
    .distinct()
)


# ------------------------------------------------------------
# Check whether every encounter has at least one claim
# ------------------------------------------------------------

claim_encounter_keys = (
    claim_keys
    .select(
        "encounter_id"
    )
    .distinct()
)

encounters_without_claims = (
    encounter_keys.alias("encounters")
    .join(
        claim_encounter_keys.alias("claims"),
        F.col("encounters.encounter_id")
        ==
        F.col("claims.encounter_id"),
        "left_anti"
    )
)

encounters_without_claim_count = (
    encounters_without_claims.count()
)


# ------------------------------------------------------------
# Check whether every claim has at least one transaction
# ------------------------------------------------------------

distinct_claim_keys = (
    claim_keys
    .select(
        "claim_id"
    )
    .distinct()
)

claims_without_transactions = (
    distinct_claim_keys.alias("claims")
    .join(
        transaction_claim_keys.alias("transactions"),
        F.col("claims.claim_id")
        ==
        F.col("transactions.claim_id"),
        "left_anti"
    )
)

claims_without_transaction_count = (
    claims_without_transactions.count()
)


# ------------------------------------------------------------
# Prove that encounter_id is nonunique in claims
# ------------------------------------------------------------

claims_per_encounter = (
    claim_keys
    .groupBy(
        "encounter_id"
    )
    .agg(
        F.count("*").alias(
            "claim_count"
        )
    )
)

encounters_with_multiple_claims = (
    claims_per_encounter
    .filter(
        F.col("claim_count") > 1
    )
    .count()
)

maximum_claims_per_encounter = (
    claims_per_encounter
    .agg(
        F.max("claim_count").alias(
            "maximum_claims"
        )
    )
    .first()["maximum_claims"]
)


# ------------------------------------------------------------
# Profile transactions per claim
# ------------------------------------------------------------

transactions_per_claim = (
    spark.table(
        f"{SILVER_SCHEMA}.claims_transactions"
    )
    .groupBy(
        "claim_id"
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        )
    )
)

transaction_cardinality = (
    transactions_per_claim
    .agg(
        F.min("transaction_count").alias(
            "minimum_transactions"
        ),
        F.avg("transaction_count").alias(
            "average_transactions"
        ),
        F.max("transaction_count").alias(
            "maximum_transactions"
        )
    )
    .first()
)

median_transactions = (
    transactions_per_claim
    .approxQuantile(
        "transaction_count",
        [0.5],
        0.001
    )[0]
)


# Create a one-row summary without defining another manual schema
hierarchy_coverage_audit = (
    spark.range(1)
    .select(
        F.lit(
            encounters_without_claim_count
        ).alias(
            "encounters_without_claims"
        ),

        F.lit(
            claims_without_transaction_count
        ).alias(
            "claims_without_transactions"
        ),

        F.lit(
            encounters_with_multiple_claims
        ).alias(
            "encounters_with_multiple_claims"
        ),

        F.lit(
            maximum_claims_per_encounter
        ).alias(
            "maximum_claims_per_encounter"
        ),

        F.lit(
            transaction_cardinality[
                "minimum_transactions"
            ]
        ).alias(
            "minimum_transactions_per_claim"
        ),

        F.lit(
            median_transactions
        ).alias(
            "median_transactions_per_claim"
        ),

        F.lit(
            transaction_cardinality[
                "average_transactions"
            ]
        ).alias(
            "average_transactions_per_claim"
        ),

        F.lit(
            transaction_cardinality[
                "maximum_transactions"
            ]
        ).alias(
            "maximum_transactions_per_claim"
        ),

        F.current_timestamp().alias(
            "audit_timestamp"
        )
    )
)

display(
    hierarchy_coverage_audit
)


# Enforce the expected relationship rules
assert encounters_without_claim_count == 0, (
    "One or more encounters do not have a claim header."
)

assert claims_without_transaction_count == 0, (
    "One or more claims do not have a transaction."
)

assert encounters_with_multiple_claims > 0, (
    "Expected encounter_id to be nonunique in claims."
)

print(
    "Every encounter has at least one claim header."
)

print(
    "Every claim has at least one transaction."
)

print(
    "claims.encounter_id is nonunique and remains "
    "a foreign key, not the claim primary key."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f78c1092-67b1-4835-9ef4-6a2767f43b60)

Every encounter has at least one claim header.
Every claim has at least one transaction.
claims.encounter_id is nonunique and remains a foreign key, not the claim primary key.


In [63]:
# ============================================================
# 7. VALIDATE TEMPORAL INTEGRITY
# ============================================================

encounters_df = spark.table(
    f"{SILVER_SCHEMA}.encounters"
)

payer_transitions_df = spark.table(
    f"{SILVER_SCHEMA}.payer_transitions"
)

claims_df = spark.table(
    f"{SILVER_SCHEMA}.claims"
)

transactions_df = spark.table(
    f"{SILVER_SCHEMA}.claims_transactions"
)


# ------------------------------------------------------------
# Encounter start/end sequencing
# ------------------------------------------------------------

encounters_end_before_start = (
    encounters_df
    .filter(
        F.col("encounter_end_timestamp")
        <
        F.col("encounter_start_timestamp")
    )
    .count()
)


# ------------------------------------------------------------
# Coverage start/end sequencing
# ------------------------------------------------------------

coverage_end_before_start = (
    payer_transitions_df
    .filter(
        F.col("coverage_end_timestamp")
        <
        F.col("coverage_start_timestamp")
    )
    .count()
)


# ------------------------------------------------------------
# Claim illness/service sequencing
# ------------------------------------------------------------

claim_illness_after_service = (
    claims_df
    .filter(
        F.col("current_illness_timestamp")
        >
        F.col("service_timestamp")
    )
    .count()
)


# ------------------------------------------------------------
# Transaction service-interval sequencing after normalization
# ------------------------------------------------------------

transaction_end_before_start = (
    transactions_df
    .filter(
        F.col("service_end_timestamp")
        <
        F.col("service_start_timestamp")
    )
    .count()
)

negative_transaction_durations = (
    transactions_df
    .filter(
        F.col("service_duration_minutes") < 0
    )
    .count()
)


# ------------------------------------------------------------
# Count documented transaction source anomalies
# ------------------------------------------------------------

transaction_epoch_placeholders = (
    transactions_df
    .filter(
        F.col("service_interval_quality")
        ==
        "epoch placeholder normalized to null"
    )
    .count()
)

transaction_invalid_intervals_documented = (
    transactions_df
    .filter(
        F.col("service_interval_quality")
        ==
        "end before start normalized to null"
    )
    .count()
)


temporal_integrity_audit = (
    spark.range(1)
    .select(
        F.lit(
            encounters_end_before_start
        ).alias(
            "encounters_end_before_start"
        ),

        F.lit(
            coverage_end_before_start
        ).alias(
            "coverage_end_before_start"
        ),

        F.lit(
            claim_illness_after_service
        ).alias(
            "claim_illness_after_service"
        ),

        F.lit(
            transaction_end_before_start
        ).alias(
            "transaction_end_before_start_after_normalization"
        ),

        F.lit(
            negative_transaction_durations
        ).alias(
            "negative_transaction_durations"
        ),

        F.lit(
            transaction_epoch_placeholders
        ).alias(
            "documented_epoch_placeholders"
        ),

        F.lit(
            transaction_invalid_intervals_documented
        ).alias(
            "documented_invalid_transaction_intervals"
        ),

        F.current_timestamp().alias(
            "audit_timestamp"
        )
    )
)

display(
    temporal_integrity_audit
)


assert encounters_end_before_start == 0, (
    "One or more encounters end before they start."
)

assert coverage_end_before_start == 0, (
    "One or more coverage intervals end before they start."
)

assert claim_illness_after_service == 0, (
    "One or more claim illness timestamps occur "
    "after the service timestamp."
)

assert transaction_end_before_start == 0, (
    "Invalid transaction intervals remain after normalization."
)

assert negative_transaction_durations == 0, (
    "Negative transaction durations remain."
)

assert transaction_epoch_placeholders == 71066, (
    "The documented epoch-placeholder count changed."
)

assert transaction_invalid_intervals_documented == 22, (
    "The documented invalid transaction-interval count changed."
)

print(
    "Encounter, coverage, claim and transaction "
    "temporal-integrity validations passed."
)

print(
    "Transaction source anomalies remain preserved "
    "through explicit quality classifications."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 65, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8088199f-ff5d-427c-a875-1eb89990456e)

Encounter, coverage, claim and transaction temporal-integrity validations passed.
Transaction source anomalies remain preserved through explicit quality classifications.


In [64]:
# ============================================================
# 8. VALIDATE FINANCIAL LINEAGE AND SOURCE ANOMALIES
# ============================================================

BRONZE_PATH = "Files/bronze/synthea"

# Read only the fields needed for independent lineage testing
bronze_transaction_financials = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(
        f"{BRONZE_PATH}/claims_transactions.csv"
    )
    .select(
        F.col("ID").alias(
            "bronze_transaction_id"
        ),

        # PAYMENTS is the source for payment_amount
        F.trim(
            F.col("PAYMENTS")
        ).cast(
            T.DecimalType(18, 2)
        ).alias(
            "expected_payment_amount"
        ),

        # AMOUNT is the source for transaction_amount
        F.trim(
            F.col("AMOUNT")
        ).cast(
            T.DecimalType(18, 2)
        ).alias(
            "expected_transaction_amount"
        )
    )
)


silver_transaction_financials = (
    transactions_df
    .select(
        "transaction_id",
        "transaction_type",
        "transaction_amount",
        "payment_amount",
        "source_patient_insurance_id",
        "patient_insurance_id_relationship_status"
    )
)


financial_lineage_comparison = (
    silver_transaction_financials.alias("silver")
    .join(
        bronze_transaction_financials.alias("bronze"),
        F.col("silver.transaction_id")
        ==
        F.col("bronze.bronze_transaction_id"),
        "inner"
    )
)


joined_transaction_count = (
    financial_lineage_comparison.count()
)


# Verify payment_amount came from PAYMENTS
payment_lineage_mismatches = (
    financial_lineage_comparison
    .filter(
        ~F.col("silver.payment_amount").eqNullSafe(
            F.col("bronze.expected_payment_amount")
        )
    )
    .count()
)


# Verify transaction_amount came from AMOUNT
transaction_amount_lineage_mismatches = (
    financial_lineage_comparison
    .filter(
        ~F.col("silver.transaction_amount").eqNullSafe(
            F.col("bronze.expected_transaction_amount")
        )
    )
    .count()
)


# Confirm all PAYMENT transactions have a payment value
payment_transaction_count = (
    financial_lineage_comparison
    .filter(
        F.col("transaction_type") == "PAYMENT"
    )
    .count()
)

payment_rows_with_null_payment = (
    financial_lineage_comparison
    .filter(
        (F.col("transaction_type") == "PAYMENT")
        &
        F.col("payment_amount").isNull()
    )
    .count()
)


# Demonstrate why AMOUNT cannot be used as payment
payment_rows_with_null_amount_but_valid_payment = (
    financial_lineage_comparison
    .filter(
        (F.col("transaction_type") == "PAYMENT")
        &
        F.col("transaction_amount").isNull()
        &
        F.col("payment_amount").isNotNull()
    )
    .count()
)


# Confirm the PATIENTINSURANCEID anomaly documentation
incorrect_insurance_relationship_labels = (
    transactions_df
    .filter(
        ~F.col(
            "patient_insurance_id_relationship_status"
        ).eqNullSafe(
            "source anomaly - not validated as payer relationship"
        )
    )
    .count()
)


# Confirm self-pay/uninsured claim classification
self_pay_claim_count = (
    claims_df
    .filter(
        F.col("primary_payer_id").isNull()
    )
    .count()
)

self_pay_classification_mismatches = (
    claims_df
    .filter(
        (
            F.col("primary_payer_id").isNull()
            &
            ~F.col("claim_coverage_category").eqNullSafe(
                "self-pay or uninsured"
            )
        )
        |
        (
            F.col("primary_payer_id").isNotNull()
            &
            ~F.col("claim_coverage_category").eqNullSafe(
                "insured"
            )
        )
    )
    .count()
)

financial_lineage_audit = (
    spark.range(1)
    .select(
        F.lit(
            joined_transaction_count
        ).alias(
            "joined_transaction_count"
        ),

        F.lit(
            payment_lineage_mismatches
        ).alias(
            "payment_lineage_mismatches"
        ),

        F.lit(
            transaction_amount_lineage_mismatches
        ).alias(
            "transaction_amount_lineage_mismatches"
        ),

        F.lit(
            payment_transaction_count
        ).alias(
            "payment_transaction_rows"
        ),

        F.lit(
            payment_rows_with_null_payment
        ).alias(
            "payment_rows_with_null_payment"
        ),

        F.lit(
            payment_rows_with_null_amount_but_valid_payment
        ).alias(
            "payment_rows_with_null_amount_but_valid_payment"
        ),

        F.lit(
            incorrect_insurance_relationship_labels
        ).alias(
            "incorrect_insurance_anomaly_labels"
        ),

        F.lit(
            self_pay_claim_count
        ).alias(
            "self_pay_or_uninsured_claims"
        ),

        F.lit(
            self_pay_classification_mismatches
        ).alias(
            "coverage_classification_mismatches"
        ),

        F.current_timestamp().alias(
            "audit_timestamp"
        )
    )
)

display(
    financial_lineage_audit
)


assert joined_transaction_count == 711238
assert payment_lineage_mismatches == 0
assert transaction_amount_lineage_mismatches == 0
assert payment_transaction_count == 252335
assert payment_rows_with_null_payment == 0

assert (
    payment_rows_with_null_amount_but_valid_payment
    ==
    252335
), (
    "The relationship between AMOUNT and PAYMENTS changed."
)

assert incorrect_insurance_relationship_labels == 0
assert self_pay_claim_count == 25967
assert self_pay_classification_mismatches == 0, (
    f"Found {self_pay_classification_mismatches:,} "
    "claims with incorrect or missing coverage classifications."
)


print(
    "Financial source-lineage validations passed."
)

print(
    "Silver payment_amount is sourced from PAYMENTS, "
    "not AMOUNT."
)

print(
    "PATIENTINSURANCEID remains a documented source anomaly "
    "and is not modeled as a payer relationship."
)

print(
    "Claim payer classifications passed."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b0539962-c08a-4f1d-a454-e77ae237e889)

Financial source-lineage validations passed.
Silver payment_amount is sourced from PAYMENTS, not AMOUNT.
PATIENTINSURANCEID remains a documented source anomaly and is not modeled as a payer relationship.
Claim payer classifications passed.


In [65]:
# ============================================================
# 9. VALIDATE FINANCIAL INTEGRITY
# ============================================================

patients_df = spark.table(
    f"{SILVER_SCHEMA}.patients"
)

encounters_df = spark.table(
    f"{SILVER_SCHEMA}.encounters"
)

claims_df = spark.table(
    f"{SILVER_SCHEMA}.claims"
)

transactions_df = spark.table(
    f"{SILVER_SCHEMA}.claims_transactions"
)


# ------------------------------------------------------------
# Patient financial checks
# ------------------------------------------------------------

negative_patient_healthcare_expenses = (
    patients_df
    .filter(
        F.col("healthcare_expenses") < 0
    )
    .count()
)

negative_patient_healthcare_coverage = (
    patients_df
    .filter(
        F.col("healthcare_coverage") < 0
    )
    .count()
)


# ------------------------------------------------------------
# Encounter financial checks
# ------------------------------------------------------------

negative_base_encounter_costs = (
    encounters_df
    .filter(
        F.col("base_encounter_cost") < 0
    )
    .count()
)

negative_total_claim_costs = (
    encounters_df
    .filter(
        F.col("total_claim_cost") < 0
    )
    .count()
)

negative_encounter_payer_coverage = (
    encounters_df
    .filter(
        F.col("payer_coverage") < 0
    )
    .count()
)

negative_patient_responsibility = (
    encounters_df
    .filter(
        F.col("patient_responsibility") < 0
    )
    .count()
)

encounter_responsibility_mismatches = (
    encounters_df
    .filter(
        ~F.col(
            "patient_responsibility"
        ).eqNullSafe(
            (
                F.coalesce(
                    F.col("total_claim_cost"),
                    F.lit(0).cast(
                        T.DecimalType(18, 2)
                    )
                )
                -
                F.coalesce(
                    F.col("payer_coverage"),
                    F.lit(0).cast(
                        T.DecimalType(18, 2)
                    )
                )
            ).cast(
                T.DecimalType(18, 2)
            )
        )
    )
    .count()
)


# ------------------------------------------------------------
# Claim outstanding-balance checks
# ------------------------------------------------------------

negative_primary_outstanding = (
    claims_df
    .filter(
        F.col("primary_outstanding_amount") < 0
    )
    .count()
)

negative_secondary_outstanding = (
    claims_df
    .filter(
        F.col("secondary_outstanding_amount") < 0
    )
    .count()
)

negative_patient_outstanding = (
    claims_df
    .filter(
        F.col("patient_outstanding_amount") < 0
    )
    .count()
)

negative_total_outstanding = (
    claims_df
    .filter(
        F.col("total_outstanding_amount") < 0
    )
    .count()
)

claim_total_outstanding_mismatches = (
    claims_df
    .filter(
        ~F.col(
            "total_outstanding_amount"
        ).eqNullSafe(
            (
                F.coalesce(
                    F.col("primary_outstanding_amount"),
                    F.lit(0).cast(
                        T.DecimalType(18, 2)
                    )
                )
                +
                F.coalesce(
                    F.col("secondary_outstanding_amount"),
                    F.lit(0).cast(
                        T.DecimalType(18, 2)
                    )
                )
                +
                F.coalesce(
                    F.col("patient_outstanding_amount"),
                    F.lit(0).cast(
                        T.DecimalType(18, 2)
                    )
                )
            ).cast(
                T.DecimalType(18, 2)
            )
        )
    )
    .count()
)


# ------------------------------------------------------------
# Transaction financial checks
# ------------------------------------------------------------

transaction_financial_columns = [
    "transaction_amount",
    "unit_amount",
    "payment_amount",
    "adjustment_amount",
    "transfer_amount",
    "outstanding_amount"
]

negative_transaction_values = {}

for column_name in transaction_financial_columns:

    negative_transaction_values[column_name] = (
        transactions_df
        .filter(
            F.col(column_name) < 0
        )
        .count()
    )

negative_transaction_units = (
    transactions_df
    .filter(
        F.col("units") < 0
    )
    .count()
)


# ------------------------------------------------------------
# Create the financial-integrity summary
# ------------------------------------------------------------

financial_integrity_audit = (
    spark.range(1)
    .select(
        F.lit(
            negative_patient_healthcare_expenses
        ).alias(
            "negative_patient_healthcare_expenses"
        ),

        F.lit(
            negative_patient_healthcare_coverage
        ).alias(
            "negative_patient_healthcare_coverage"
        ),

        F.lit(
            negative_base_encounter_costs
        ).alias(
            "negative_base_encounter_costs"
        ),

        F.lit(
            negative_total_claim_costs
        ).alias(
            "negative_total_claim_costs"
        ),

        F.lit(
            negative_encounter_payer_coverage
        ).alias(
            "negative_encounter_payer_coverage"
        ),

        F.lit(
            negative_patient_responsibility
        ).alias(
            "negative_patient_responsibility"
        ),

        F.lit(
            encounter_responsibility_mismatches
        ).alias(
            "encounter_responsibility_mismatches"
        ),

        F.lit(
            negative_primary_outstanding
        ).alias(
            "negative_primary_outstanding"
        ),

        F.lit(
            negative_secondary_outstanding
        ).alias(
            "negative_secondary_outstanding"
        ),

        F.lit(
            negative_patient_outstanding
        ).alias(
            "negative_patient_outstanding"
        ),

        F.lit(
            negative_total_outstanding
        ).alias(
            "negative_total_outstanding"
        ),

        F.lit(
            claim_total_outstanding_mismatches
        ).alias(
            "claim_total_outstanding_mismatches"
        ),

        F.lit(
            negative_transaction_values[
                "transaction_amount"
            ]
        ).alias(
            "negative_transaction_amounts"
        ),

        F.lit(
            negative_transaction_values[
                "unit_amount"
            ]
        ).alias(
            "negative_unit_amounts"
        ),

        F.lit(
            negative_transaction_values[
                "payment_amount"
            ]
        ).alias(
            "negative_payment_amounts"
        ),

        F.lit(
            negative_transaction_values[
                "adjustment_amount"
            ]
        ).alias(
            "negative_adjustment_amounts"
        ),

        F.lit(
            negative_transaction_values[
                "transfer_amount"
            ]
        ).alias(
            "negative_transfer_amounts"
        ),

        F.lit(
            negative_transaction_values[
                "outstanding_amount"
            ]
        ).alias(
            "negative_transaction_outstanding_amounts"
        ),

        F.lit(
            negative_transaction_units
        ).alias(
            "negative_transaction_units"
        ),

        F.current_timestamp().alias(
            "audit_timestamp"
        )
    )
)

display(
    financial_integrity_audit
)


# Every audit value except the timestamp must equal zero
financial_failure_count = sum([
    negative_patient_healthcare_expenses,
    negative_patient_healthcare_coverage,
    negative_base_encounter_costs,
    negative_total_claim_costs,
    negative_encounter_payer_coverage,
    negative_patient_responsibility,
    encounter_responsibility_mismatches,
    negative_primary_outstanding,
    negative_secondary_outstanding,
    negative_patient_outstanding,
    negative_total_outstanding,
    claim_total_outstanding_mismatches,
    *negative_transaction_values.values(),
    negative_transaction_units
])

if financial_failure_count > 0:
    raise ValueError(
        f"Financial-integrity validation found "
        f"{financial_failure_count:,} failed records."
    )

print(
    "All Silver financial-integrity validations passed."
)

print(
    "Derived encounter responsibility and claim outstanding "
    "balances reconcile with their component values."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 67, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a679d67-4ece-4d92-a13d-f740c12f1a0d)

All Silver financial-integrity validations passed.
Derived encounter responsibility and claim outstanding balances reconcile with their component values.


In [66]:
# ============================================================
# 10. VALIDATE RECORD HASHES AND AUDIT METADATA
# ============================================================

REQUIRED_AUDIT_COLUMNS = [
    "record_hash",
    "source_system",
    "source_file",
    "silver_processed_timestamp"
]

metadata_audit_results = []

for table_name in SILVER_TABLES:

    full_table_name = (
        f"{SILVER_SCHEMA}.{table_name}"
    )

    table_df = spark.table(
        full_table_name
    )

    table_columns = set(
        table_df.columns
    )

    missing_audit_columns = [
        column_name
        for column_name in REQUIRED_AUDIT_COLUMNS
        if column_name not in table_columns
    ]

    row_count = table_df.count()

    if len(missing_audit_columns) == 0:

        metadata_summary = (
            table_df
            .agg(
                F.sum(
                    F.when(
                        F.col("record_hash").isNull(),
                        1
                    ).otherwise(0)
                ).alias(
                    "null_record_hashes"
                ),

                F.sum(
                    F.when(
                        F.length("record_hash") != 64,
                        1
                    ).otherwise(0)
                ).alias(
                    "invalid_hash_lengths"
                ),

                F.sum(
                    F.when(
                        F.col("source_system").isNull(),
                        1
                    ).otherwise(0)
                ).alias(
                    "null_source_systems"
                ),

                F.sum(
                    F.when(
                        F.col("source_file").isNull(),
                        1
                    ).otherwise(0)
                ).alias(
                    "null_source_files"
                ),

                F.sum(
                    F.when(
                        F.col(
                            "silver_processed_timestamp"
                        ).isNull(),
                        1
                    ).otherwise(0)
                ).alias(
                    "null_processed_timestamps"
                )
            )
            .first()
        )

        null_record_hashes = (
            metadata_summary["null_record_hashes"]
        )

        invalid_hash_lengths = (
            metadata_summary["invalid_hash_lengths"]
        )

        null_source_systems = (
            metadata_summary["null_source_systems"]
        )

        null_source_files = (
            metadata_summary["null_source_files"]
        )

        null_processed_timestamps = (
            metadata_summary[
                "null_processed_timestamps"
            ]
        )

        audit_status = (
            "PASS"
            if (
                null_record_hashes == 0
                and
                invalid_hash_lengths == 0
                and
                null_source_systems == 0
                and
                null_source_files == 0
                and
                null_processed_timestamps == 0
            )
            else "FAIL"
        )

    else:
        null_record_hashes = None
        invalid_hash_lengths = None
        null_source_systems = None
        null_source_files = None
        null_processed_timestamps = None
        audit_status = "MISSING COLUMNS"

    metadata_audit_results.append(
        (
            full_table_name,
            row_count,
            ", ".join(missing_audit_columns),
            null_record_hashes,
            invalid_hash_lengths,
            null_source_systems,
            null_source_files,
            null_processed_timestamps,
            audit_status
        )
    )


metadata_audit_schema = T.StructType([
    T.StructField("table_name", T.StringType(), False),
    T.StructField("row_count", T.LongType(), False),
    T.StructField("missing_audit_columns", T.StringType(), False),
    T.StructField("null_record_hashes", T.LongType(), True),
    T.StructField("invalid_hash_lengths", T.LongType(), True),
    T.StructField("null_source_systems", T.LongType(), True),
    T.StructField("null_source_files", T.LongType(), True),
    T.StructField("null_processed_timestamps", T.LongType(), True),
    T.StructField("audit_status", T.StringType(), False)
])


metadata_audit = (
    spark.createDataFrame(
        metadata_audit_results,
        schema=metadata_audit_schema
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
)

display(
    metadata_audit
    .orderBy("table_name")
)


failed_metadata_table_count = (
    metadata_audit
    .filter(
        F.col("audit_status") != "PASS"
    )
    .count()
)

if failed_metadata_table_count > 0:
    raise ValueError(
        f"{failed_metadata_table_count} Silver tables "
        "failed record-hash or metadata validation."
    )

print(
    "Record hashes and audit metadata passed "
    "for all six Silver tables."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 68, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7ff87e9c-3422-4d24-a63f-9155fbd0bda5)

Record hashes and audit metadata passed for all six Silver tables.


In [67]:
# ============================================================
# 11. CREATE CONSOLIDATED SILVER AUDIT SUMMARY
# ============================================================

import uuid

AUDIT_RUN_ID = str(
    uuid.uuid4()
)

# Count only genuine temporal failures.
# Documented source anomalies are not failures.
temporal_failure_count = (
    encounters_end_before_start
    +
    coverage_end_before_start
    +
    claim_illness_after_service 
    +
    transaction_end_before_start
    +
    negative_transaction_durations
)

# Count financial-lineage failures
financial_lineage_failure_count = (
    payment_lineage_mismatches
    +
    transaction_amount_lineage_mismatches
    +
    payment_rows_with_null_payment
    +
    incorrect_insurance_relationship_labels
    +
    self_pay_classification_mismatches
)

# Count hierarchy coverage failures
hierarchy_failure_count = (
    encounters_without_claim_count
    +
    claims_without_transaction_count
)

audit_summary_records = [
    (
        "row_count_reconciliation",
        "All six Silver tables match expected row counts",
        failed_table_count,
        0,
        (
            "PASS"
            if failed_table_count == 0
            else "FAIL"
        )
    ),
    (
        "primary_key_integrity",
        "Five defined primary keys are complete and unique",
        failed_primary_key_count,
        0,
        (
            "PASS"
            if failed_primary_key_count == 0
            else "FAIL"
        )
    ),
    (
        "exact_duplicate_validation",
        "Six Silver tables contain no exact duplicate rows",
        failed_duplicate_table_count,
        0,
        (
            "PASS"
            if failed_duplicate_table_count == 0
            else "FAIL"
        )
    ),
    (
        "foreign_key_integrity",
        "All intended foreign-key relationships are valid",
        failed_foreign_key_count,
        0,
        (
            "PASS"
            if failed_foreign_key_count == 0
            else "FAIL"
        )
    ),
    (
        "hierarchy_coverage",
        "Every encounter has a claim and every claim has a transaction",
        hierarchy_failure_count,
        0,
        (
            "PASS"
            if hierarchy_failure_count == 0
            else "FAIL"
        )
    ),
    (
        "temporal_integrity",
        "No unresolved invalid temporal sequences remain",
        temporal_failure_count,
        0,
        (
            "PASS"
            if temporal_failure_count == 0
            else "FAIL"
        )
    ),
    (
        "financial_source_lineage",
        "PAYMENTS and AMOUNT map to the correct Silver fields",
        financial_lineage_failure_count,
        0,
        (
            "PASS"
            if financial_lineage_failure_count == 0
            else "FAIL"
        )
    ),
    (
        "financial_integrity",
        "Financial values and derived balances are valid",
        financial_failure_count,
        0,
        (
            "PASS"
            if financial_failure_count == 0
            else "FAIL"
        )
    ),
    (
        "hash_and_metadata_integrity",
        "All tables contain valid hashes and audit metadata",
        failed_metadata_table_count,
        0,
        (
            "PASS"
            if failed_metadata_table_count == 0
            else "FAIL"
        )
    )
]


audit_summary_schema = T.StructType([
    T.StructField(
        "audit_category",
        T.StringType(),
        False
    ),
    T.StructField(
        "audit_description",
        T.StringType(),
        False
    ),
    T.StructField(
        "actual_failure_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "expected_failure_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "audit_status",
        T.StringType(),
        False
    )
])


silver_audit_summary = (
    spark.createDataFrame(
        audit_summary_records,
        schema=audit_summary_schema
    )
    .withColumn(
        "audit_run_id",
        F.lit(AUDIT_RUN_ID)
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
    .select(
        "audit_run_id",
        "audit_timestamp",
        "audit_category",
        "audit_description",
        "actual_failure_count",
        "expected_failure_count",
        "audit_status"
    )
)

display(
    silver_audit_summary
    .orderBy("audit_category")
)


failed_audit_category_count = (
    silver_audit_summary
    .filter(
        F.col("audit_status") != "PASS"
    )
    .count()
)

if failed_audit_category_count > 0:
    raise ValueError(
        f"{failed_audit_category_count} consolidated "
        "Silver audit categories failed."
    )

print(
    f"Silver audit run completed successfully: "
    f"{AUDIT_RUN_ID}"
)

print(
    "All nine consolidated audit categories passed."
)

StatementMeta(, 3858e519-6634-4aad-98b8-38de53bada83, 69, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c0777b45-488b-4822-82a4-57d57330b7df)

Silver audit run completed successfully: 0f665361-b481-4680-9fe2-f27916717490
All nine consolidated audit categories passed.
